In [ ]:
# CAFE — configuration cell. Run this first.
#
# All paths below are resolved relative to the repository root, so the notebook
# runs unchanged on any machine. Override with the CAFE_ROOT environment
# variable if the weights/data live outside the repo.
import os
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get("CAFE_ROOT", Path.cwd().parent)).resolve()
assert (PROJECT_ROOT / "runs").is_dir(), (
    f"CAFE_ROOT does not look like the repo root: {PROJECT_ROOT}"
)
print("PROJECT_ROOT =", PROJECT_ROOT)


In [ ]:
import torch, os
assert torch.cuda.is_available(), "No CUDA"

WEIGHTS = {
    "baseline"  : f"{PROJECT_ROOT}/weights/P0/classifier.pt",
    "pruned_10" : f"{PROJECT_ROOT}/weights/P10/classifier.pt",
    "pruned_15" : f"{PROJECT_ROOT}/weights/P15/classifier.pt",
}
TEST_DIR = f"{PROJECT_ROOT}/data/FER2013/test"

for name, path in WEIGHTS.items():
    assert os.path.isfile(path), f"Missing: {name}"
print("OK")

In [ ]:
import os, io, contextlib, torch
import pandas as pd
from ultralytics import YOLO
from torchinfo import summary as ti_summary

DEVICE = torch.device("cuda:0")
torch.cuda.set_device(DEVICE)
IMGSZ, BATCH = 224, 1

models, profiles = {}, []
for name, weight_path in WEIGHTS.items():
    model = YOLO(weight_path, task="classify")
    model.model.to(DEVICE).eval()

    disk_mb = os.path.getsize(weight_path) / 1e6
    vram_mb = sum(p.numel() * p.element_size() for p in model.model.parameters()) / 1e6

    with contextlib.redirect_stdout(io.StringIO()):
        ti = ti_summary(model.model, input_size=(BATCH, 3, IMGSZ, IMGSZ), device=DEVICE, verbose=0)

    profiles.append({
        "Model": name,
        "Disk Size (MB)": round(disk_mb, 2),
        "VRAM fp32 (MB)": round(vram_mb, 2),
        "Params (M)": round(ti.total_params / 1e6, 3),
        "GFLOPs": round(ti.total_mult_adds / 1e9, 4),
    })
    models[name] = model

df_profile = pd.DataFrame(profiles).set_index("Model")
print(df_profile.to_string())

In [ ]:
import shutil
from pathlib import Path

IMGSZ, BATCH = 224, 1
DATA_ROOT = Path(f"{PROJECT_ROOT}/data/FER2013")
ENGINE_DIR = Path("/tmp/trt_engines")
ENGINE_DIR.mkdir(exist_ok=True)

# val/ symlink → test/ (ultralytics calibrator defaults to 'val' split)
val_link = DATA_ROOT / "val"
if val_link.exists() or val_link.is_symlink():
    val_link.unlink()
val_link.symlink_to((DATA_ROOT / "test").resolve())

ENGINE_PATHS = {}
for name, model in models.items():
    out_engine = ENGINE_DIR / f"{name}.engine"
    exported = model.export(
        format="engine", imgsz=IMGSZ, batch=BATCH, int8=True,
        data=str(DATA_ROOT), split="val", device=0,
        half=False, simplify=True, verbose=False, workspace=4,
    )
    exported_path = Path(str(exported)) if exported else None

    if exported_path and exported_path.is_file():
        if exported_path != out_engine:
            shutil.copy2(exported_path, out_engine)
        ENGINE_PATHS[name] = out_engine
    else:
        candidates = list(Path(WEIGHTS[name]).parent.glob("*.engine"))
        if candidates:
            shutil.copy2(candidates[0], out_engine)
            ENGINE_PATHS[name] = out_engine
        else:
            print(f"Export failed: {name}")

print(f"{len(ENGINE_PATHS)}/{len(WEIGHTS)} engines ready → {ENGINE_DIR}")

In [ ]:
import torch, pandas as pd
from pathlib import Path
from ultralytics import YOLO

TEST_DIR = Path(f"{PROJECT_ROOT}/data/FER2013/test")
IMGSZ = 224
CLASSES = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
CLS2IDX = {c: i for i, c in enumerate(CLASSES)}

image_list = []
for cls_name in CLASSES:
    cls_dir = TEST_DIR / cls_name
    if cls_dir.is_dir():
        for img_path in sorted(cls_dir.iterdir()):
            if img_path.suffix.lower() in ('.jpg', '.jpeg', '.png'):
                image_list.append((img_path, CLS2IDX[cls_name]))

all_results = {}
for name, eng_path in ENGINE_PATHS.items():
    trt_model = YOLO(str(eng_path), task="classify")
    for wp, _ in image_list[:10]:
        trt_model.predict(source=str(wp), imgsz=IMGSZ, device=0, verbose=False)
    torch.cuda.synchronize()

    records = []
    for img_path, true_idx in image_list:
        results = trt_model.predict(source=str(img_path), imgsz=IMGSZ, device=0, verbose=False)
        torch.cuda.synchronize()
        speed = results[0].speed
        pred_idx = int(results[0].probs.top1)
        records.append({
            "correct": int(pred_idx == true_idx),
            "pre_ms": speed["preprocess"],
            "inf_ms": speed["inference"],
            "post_ms": speed["postprocess"],
        })
    all_results[name] = records

summary_rows = []
for name, records in all_results.items():
    n = len(records)
    summary_rows.append({
        "Model": name,
        "Top1 Acc (%)": round(sum(r["correct"] for r in records) / n * 100, 2),
        "Avg Pre (ms)": round(sum(r["pre_ms"] for r in records) / n, 3),
        "Avg Inf (ms)": round(sum(r["inf_ms"] for r in records) / n, 3),
        "Avg Post(ms)": round(sum(r["post_ms"] for r in records) / n, 3),
        "Images": n,
    })

df_summary = pd.DataFrame(summary_rows).set_index("Model")
print(df_summary.to_string())

In [ ]:
import torch, time, cv2, numpy as np, pandas as pd, contextlib, shutil
from pathlib import Path
from ultralytics import YOLO

DEVICE = torch.device("cuda:0")
IMGSZ, BATCH = 224, 1
CLASSES = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
DATA_ROOT = Path(f"{PROJECT_ROOT}/data/FER2013")
ENGINE_DIR = Path("/tmp/trt_engines")
ENGINE_DIR.mkdir(exist_ok=True)

# ── Export FP16 engines ─────────────────────────────────────────
val_link = DATA_ROOT / "val"
if not val_link.exists():
    val_link.symlink_to((DATA_ROOT / "test").resolve())

FP16_ENGINE_PATHS = {}
for name, weight_path in WEIGHTS.items():
    out_engine = ENGINE_DIR / f"{name}_fp16.engine"
    exported = YOLO(weight_path, task="classify").export(
        format="engine", imgsz=IMGSZ, batch=BATCH, half=True, int8=False,
        data=str(DATA_ROOT), split="val", device=0,
        simplify=True, verbose=False, workspace=4,
    )
    exported_path = Path(str(exported)) if exported else None
    if exported_path and exported_path.is_file():
        if exported_path != out_engine:
            shutil.copy2(exported_path, out_engine)
        FP16_ENGINE_PATHS[name] = out_engine
    else:
        cands = list(Path(weight_path).parent.glob("*.engine"))
        cands = [c for c in cands if "fp16" in c.name or
                 c.stat().st_mtime == max(cc.stat().st_mtime for cc in cands)]
        if cands:
            shutil.copy2(cands[0], out_engine)
            FP16_ENGINE_PATHS[name] = out_engine

# ── Inference ─────────────────────────────────────────────────────────────────
def preprocess(img_path):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMGSZ, IMGSZ), interpolation=cv2.INTER_LINEAR)
    img = np.transpose(img.astype(np.float32) / 255.0, (2, 0, 1))[None]
    return torch.from_numpy(img).to(DEVICE)

run_matrix = [(f"{n}_fp32", n, p, "FP32") for n, p in WEIGHTS.items()]
run_matrix += [(f"{n}_fp16", n, str(e), "FP16") for n, e in FP16_ENGINE_PATHS.items()]

def run_inference(model_source, image_list):
    is_engine = str(model_source).endswith(".engine")
    yolo = YOLO(model_source, task="classify")
    if not is_engine:
        yolo.model.to(DEVICE).eval()

    if is_engine:
        for img_path, _ in image_list[:10]:
            yolo.predict(source=str(img_path), imgsz=IMGSZ, device=0, verbose=False)
    else:
        with torch.no_grad():
            for img_path, _ in image_list[:10]:
                yolo.model(preprocess(img_path))
    torch.cuda.synchronize()

    records = []
    ctx = torch.no_grad() if not is_engine else contextlib.nullcontext()
    with ctx:
        for img_path, true_idx in image_list:
            if is_engine:
                results = yolo.predict(source=str(img_path), imgsz=IMGSZ, device=0, verbose=False)
                torch.cuda.synchronize()
                s = results[0].speed
                t_pre, t_inf, t_post = s["preprocess"], s["inference"], s["postprocess"]
                pred_idx = int(results[0].probs.top1)
            else:
                torch.cuda.synchronize(); t0 = time.perf_counter()
                x = preprocess(img_path)
                torch.cuda.synchronize(); t_pre = (time.perf_counter() - t0) * 1000
                t0 = time.perf_counter()
                out = yolo.model(x)
                torch.cuda.synchronize(); t_inf = (time.perf_counter() - t0) * 1000
                t0 = time.perf_counter()
                logits = out if isinstance(out, torch.Tensor) else out[0]
                pred_idx = int(torch.softmax(logits[0], dim=0).argmax().item())
                torch.cuda.synchronize(); t_post = (time.perf_counter() - t0) * 1000
            records.append({
                "correct": int(pred_idx == true_idx),
                "pre_ms": t_pre, "inf_ms": t_inf, "post_ms": t_post,
                "total_ms": t_pre + t_inf + t_post,
            })
    return records

fp32_fp16_results = {lbl: run_inference(src, image_list) for lbl, _, src, _ in run_matrix}

# ── Comparison, dropping unwanted rows ────────────────────────────────────────
HIDE = {("baseline", "FP16"), ("baseline", "INT8"),
        ("pruned_10", "FP32"), ("pruned_15", "FP32")}

def summarize(model, prec, records):
    n = len(records)
    return {
        "Model": model, "Precision": prec,
        "Top1 Acc (%)": round(sum(r["correct"] for r in records) / n * 100, 2),
        "Avg Pre (ms)": round(sum(r["pre_ms"] for r in records) / n, 3),
        "Avg Inf (ms)": round(sum(r["inf_ms"] for r in records) / n, 3),
        "Avg Post(ms)": round(sum(r["post_ms"] for r in records) / n, 3),
        "Avg Tot (ms)": round(sum(r["pre_ms"] + r["inf_ms"] + r["post_ms"] for r in records) / n, 3),
    }

comparison_rows = [summarize(n, "INT8", r) for n, r in all_results.items()]
comparison_rows += [summarize(m, p, fp32_fp16_results[lbl])
                    for lbl, m, _, p in run_matrix]
comparison_rows = [r for r in comparison_rows if (r["Model"], r["Precision"]) not in HIDE]

prec_order = ["FP32", "FP16", "INT8"]
df_compare = (
    pd.DataFrame(comparison_rows)
      .assign(Precision=lambda d: pd.Categorical(d["Precision"], categories=prec_order, ordered=True))
      .sort_values(["Model", "Precision"])
      .set_index(["Model", "Precision"])
)
print(df_compare.to_string())

In [ ]:
import os, torch, yaml
assert torch.cuda.is_available(), "No CUDA"

# ── Classifier ────────────────────────────────────────────────────────────────
WEIGHTS = {
    "baseline"  : f"{PROJECT_ROOT}/weights/P0/classifier.pt",
    "pruned_10" : f"{PROJECT_ROOT}/weights/P10/classifier.pt",
    "pruned_15" : f"{PROJECT_ROOT}/weights/P15/classifier.pt",
}
CLS_TEST_DIR = f"{PROJECT_ROOT}/data/FER2013/test"

# ── Detector ──────────────────────────────────────────────────────────────────
DET_WEIGHTS = {
    "baseline"  : f"{PROJECT_ROOT}/weights/P0/detector.pt",
    "pruned_10" : f"{PROJECT_ROOT}/weights/P10/detector.pt",
    "pruned_15" : f"{PROJECT_ROOT}/weights/P15/detector.pt",
}
DATASET_YAML  = f"{PROJECT_ROOT}/data/wider_face_w_test/widerface_yolo.yaml"
DATASET_ROOT  = f"{PROJECT_ROOT}/data/wider_face_w_test"
TRAIN_IMG_DIR = f"{DATASET_ROOT}/images/train"
TRAIN_LBL_DIR = f"{DATASET_ROOT}/labels/train"
VAL_IMG_DIR   = f"{DATASET_ROOT}/images/val"
VAL_LBL_DIR   = f"{DATASET_ROOT}/labels/val"
TEST_IMG_DIR  = f"{DATASET_ROOT}/images/test"
TEST_LBL_DIR  = f"{DATASET_ROOT}/labels/test"

for name, path in {**WEIGHTS, **DET_WEIGHTS}.items():
    assert os.path.isfile(path), f"Missing: {path}"
assert os.path.isdir(CLS_TEST_DIR), "Missing CLS_TEST_DIR"
with open(DATASET_YAML) as f:
    cfg = yaml.safe_load(f)
nc = len(cfg["names"]) if isinstance(cfg.get("names"), list) else cfg.get("nc")
assert nc == 1, f"Expected nc=1, got {nc}"
print("OK")

In [ ]:
import os, io, contextlib, torch
import pandas as pd
from ultralytics import YOLO
from torchinfo import summary as ti_summary

DEVICE = torch.device("cuda:0")
torch.cuda.set_device(DEVICE)
DET_IMGSZ, BATCH = 640, 1

det_models, det_profiles = {}, []
for name, weight_path in DET_WEIGHTS.items():
    model = YOLO(weight_path, task="detect")
    model.model.to(DEVICE).eval()

    disk_mb = os.path.getsize(weight_path) / 1e6
    vram_mb = sum(p.numel() * p.element_size() for p in model.model.parameters()) / 1e6

    with contextlib.redirect_stdout(io.StringIO()):
        ti = ti_summary(model.model, input_size=(BATCH, 3, DET_IMGSZ, DET_IMGSZ), device=DEVICE, verbose=0)

    total_params = sum(p.numel() for p in model.model.parameters())
    det_profiles.append({
        "Model": name,
        "Disk Size (MB)": round(disk_mb, 2),
        "VRAM fp32 (MB)": round(vram_mb, 2),
        "Params (M)": round(total_params / 1e6, 3),
        "GFLOPs": round(ti.total_mult_adds / 1e9, 4),
    })
    det_models[name] = model

df_det_profile = pd.DataFrame(det_profiles).set_index("Model")
print(df_det_profile.to_string())

In [ ]:
import torch, pandas as pd
from pathlib import Path
from ultralytics import YOLO

TEST_IMG_DIR = f"{PROJECT_ROOT}/data/wider_face_w_test/images/test"
DET_IMGSZ = 640
DET_ENGINE_DIR = Path("/tmp/trt_det_engines")

# Rebuild DET_ENGINE_PATHS from already-exported engines (from Cell 3)
DET_ENGINE_PATHS = {
    name: DET_ENGINE_DIR / f"{name}.engine"
    for name in DET_WEIGHTS
    if (DET_ENGINE_DIR / f"{name}.engine").is_file()
}
assert DET_ENGINE_PATHS, "No engines found in /tmp/trt_det_engines — run Cell 3 (det) first"

IMG_EXT = ('.jpg', '.jpeg', '.png')
test_image_paths = sorted(p for p in Path(TEST_IMG_DIR).rglob("*") if p.suffix.lower() in IMG_EXT)

det_all_results = {}
for name, eng_path in DET_ENGINE_PATHS.items():
    trt_model = YOLO(str(eng_path), task="detect")
    for wp in test_image_paths[:10]:
        trt_model.predict(source=str(wp), imgsz=DET_IMGSZ, device=0, verbose=False)
    torch.cuda.synchronize()

    records = []
    for img_path in test_image_paths:
        results = trt_model.predict(source=str(img_path), imgsz=DET_IMGSZ, device=0, verbose=False)
        torch.cuda.synchronize()
        speed = results[0].speed
        boxes = results[0].boxes
        n_dets = len(boxes) if boxes is not None else 0
        records.append({
            "n_dets": n_dets,
            "avg_conf": round(float(boxes.conf.mean()) if n_dets > 0 else 0.0, 4),
            "pre_ms": speed["preprocess"],
            "inf_ms": speed["inference"],
            "post_ms": speed["postprocess"],
            "total_ms": speed["preprocess"] + speed["inference"] + speed["postprocess"],
        })
    det_all_results[name] = records

summary_rows = []
for name, records in det_all_results.items():
    n = len(records)
    summary_rows.append({
        "Model": name, "Images": n,
        "Avg Dets/img": round(sum(r["n_dets"] for r in records) / n, 2),
        "Avg Pre (ms)": round(sum(r["pre_ms"] for r in records) / n, 3),
        "Avg Inf (ms)": round(sum(r["inf_ms"] for r in records) / n, 3),
        "Avg Post(ms)": round(sum(r["post_ms"] for r in records) / n, 3),
    })

df_det_int8_summary = pd.DataFrame(summary_rows).set_index("Model")
print(df_det_int8_summary.to_string())

In [ ]:
import torch, time, cv2, contextlib, numpy as np, pandas as pd
from pathlib import Path
from ultralytics import YOLO

DEVICE = torch.device("cuda:0")
DET_IMGSZ = 640
DATASET_YAML = f"{PROJECT_ROOT}/data/wider_face_w_test/widerface_yolo.yaml"
TEST_IMG_DIR = f"{PROJECT_ROOT}/data/wider_face_w_test/images/test"
DET_ENGINE_DIR = Path("/tmp/trt_det_engines")

# Rows to remove entirely
SKIP = {("baseline", "FP16"), ("baseline", "INT8"),
        ("pruned_10", "FP32"), ("pruned_15", "FP32")}

DET_FP16_ENGINE_PATHS = {n: DET_ENGINE_DIR / f"{n}_fp16.engine"
                         for n in DET_WEIGHTS if (DET_ENGINE_DIR / f"{n}_fp16.engine").is_file()}
DET_ENGINE_PATHS = {n: DET_ENGINE_DIR / f"{n}.engine"
                    for n in DET_WEIGHTS if (DET_ENGINE_DIR / f"{n}.engine").is_file()}
assert DET_FP16_ENGINE_PATHS, "No FP16 engines found — run Cell 5 (det) first"
assert DET_ENGINE_PATHS, "No INT8 engines found — run Cell 3 (det) first"

IMG_EXT = ('.jpg', '.jpeg', '.png')
test_image_paths = sorted(p for p in Path(TEST_IMG_DIR).rglob("*") if p.suffix.lower() in IMG_EXT)

def preprocess_yolo_det(img_path, imgsz=640):
    img = cv2.imread(str(img_path))
    h0, w0 = img.shape[:2]
    r = min(imgsz / h0, imgsz / w0)
    nh, nw = int(round(h0 * r)), int(round(w0 * r))
    img = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    top = (imgsz - nh) // 2; bot = imgsz - nh - top
    left = (imgsz - nw) // 2; right = imgsz - nw - left
    img = cv2.copyMakeBorder(img, top, bot, left, right, cv2.BORDER_CONSTANT, value=(114, 114, 114))
    img = np.transpose(img.astype(np.float32) / 255.0, (2, 0, 1))[None]
    return torch.from_numpy(img).to(DEVICE)

def postprocess_yolo_det(raw, conf_thr=0.25):
    if isinstance(raw, (list, tuple)):
        raw = raw[0]
    scores = raw[0][4, :]
    keep = scores >= conf_thr
    n_dets = int(keep.sum().item())
    avg_conf = float(scores[keep].mean().item()) if n_dets > 0 else 0.0
    return n_dets, avg_conf

det_run_matrix = [(f"{n}_fp32", n, p, "FP32") for n, p in DET_WEIGHTS.items()
                  if (n, "FP32") not in SKIP]
det_run_matrix += [(f"{n}_fp16", n, str(e), "FP16") for n, e in DET_FP16_ENGINE_PATHS.items()
                   if (n, "FP16") not in SKIP]

def run_det_latency(model_source, image_paths):
    is_engine = str(model_source).endswith(".engine")
    yolo = YOLO(model_source, task="detect")
    if not is_engine:
        yolo.model.to(DEVICE).eval()

    if is_engine:
        for wp in image_paths[:10]:
            yolo.predict(source=str(wp), imgsz=DET_IMGSZ, device=0, verbose=False)
    else:
        with torch.no_grad():
            for wp in image_paths[:10]:
                yolo.model(preprocess_yolo_det(wp, DET_IMGSZ))
    torch.cuda.synchronize()

    records = []
    ctx = torch.no_grad() if not is_engine else contextlib.nullcontext()
    with ctx:
        for img_path in image_paths:
            if is_engine:
                results = yolo.predict(source=str(img_path), imgsz=DET_IMGSZ, device=0, verbose=False)
                torch.cuda.synchronize()
                s = results[0].speed
                t_pre, t_inf, t_post = s["preprocess"], s["inference"], s["postprocess"]
                boxes = results[0].boxes
                n_dets = len(boxes) if boxes is not None else 0
            else:
                torch.cuda.synchronize(); t0 = time.perf_counter()
                x = preprocess_yolo_det(img_path, DET_IMGSZ)
                torch.cuda.synchronize(); t_pre = (time.perf_counter() - t0) * 1000
                t0 = time.perf_counter()
                out = yolo.model(x)
                torch.cuda.synchronize(); t_inf = (time.perf_counter() - t0) * 1000
                t0 = time.perf_counter()
                n_dets, _ = postprocess_yolo_det(out)
                torch.cuda.synchronize(); t_post = (time.perf_counter() - t0) * 1000
            records.append({
                "n_dets": n_dets,
                "pre_ms": t_pre, "inf_ms": t_inf, "post_ms": t_post,
                "total_ms": t_pre + t_inf + t_post,
            })
    return records

det_fp32_fp16_results = {lbl: run_det_latency(src, test_image_paths)
                         for lbl, _, src, _ in det_run_matrix}

# ── mAP via model.val() ───────────────────────────────────────────────────────
map_run_matrix = [(n, p, "FP32") for n, p in DET_WEIGHTS.items() if (n, "FP32") not in SKIP]
map_run_matrix += [(n, str(e), "FP16") for n, e in DET_FP16_ENGINE_PATHS.items() if (n, "FP16") not in SKIP]
map_run_matrix += [(n, str(e), "INT8") for n, e in DET_ENGINE_PATHS.items() if (n, "INT8") not in SKIP]

map_results = []
for model_name, model_source, precision in map_run_matrix:
    metrics = YOLO(model_source, task="detect").val(
        data=DATASET_YAML, split="test", imgsz=DET_IMGSZ, batch=1, device=0, verbose=False)
    map_results.append({
        "Model": model_name, "Precision": precision,
        "mAP@50": round(float(metrics.box.map50) * 100, 2),
        "mAP@50-95": round(float(metrics.box.map) * 100, 2),
        "Precision%": round(float(metrics.box.mp) * 100, 2),
        "Recall%": round(float(metrics.box.mr) * 100, 2),
    })

# ── Merge latency + mAP ───────────────────────────────────────────────────────
latency_rows = {}
for lbl, model_name, _, precision in det_run_matrix:
    records = det_fp32_fp16_results[lbl]
    n = len(records)
    latency_rows[(model_name, precision)] = {
        "Avg Pre (ms)": round(sum(r["pre_ms"] for r in records) / n, 3),
        "Avg Inf (ms)": round(sum(r["inf_ms"] for r in records) / n, 3),
        "Avg Post(ms)": round(sum(r["post_ms"] for r in records) / n, 3),
        "Avg Tot (ms)": round(sum(r["total_ms"] for r in records) / n, 3),
    }
for name, records in det_all_results.items():
    if (name, "INT8") in SKIP:
        continue
    n = len(records)
    latency_rows[(name, "INT8")] = {
        "Avg Pre (ms)": round(sum(r["pre_ms"] for r in records) / n, 3),
        "Avg Inf (ms)": round(sum(r["inf_ms"] for r in records) / n, 3),
        "Avg Post(ms)": round(sum(r["post_ms"] for r in records) / n, 3),
        "Avg Tot (ms)": round(sum(r["total_ms"] for r in records) / n, 3),
    }

comparison_rows = []
for row in map_results:
    lat = latency_rows.get((row["Model"], row["Precision"]),
                           {"Avg Pre (ms)": None, "Avg Inf (ms)": None,
                            "Avg Post(ms)": None, "Avg Tot (ms)": None})
    comparison_rows.append({**row, **lat})

prec_order = ["FP32", "FP16", "INT8"]
df_det_compare = (
    pd.DataFrame(comparison_rows)
      .assign(Precision=lambda d: pd.Categorical(d["Precision"], categories=prec_order, ordered=True))
      .sort_values(["Model", "Precision"])
      .set_index(["Model", "Precision"])
)
print(df_det_compare.to_string())

In [ ]:
import numpy as np, pandas as pd, copy
from pathlib import Path
import onnx
from onnx import shape_inference as onnx_si

def _concretize_batch(model_proto, batch=1):
    proto = copy.deepcopy(model_proto)
    for inp in proto.graph.input:
        shape = inp.type.tensor_type.shape
        if not shape or len(shape.dim) == 0:
            continue
        dim0 = shape.dim[0]
        if dim0.HasField("dim_param") or dim0.dim_value == 0:
            dim0.ClearField("dim_param")
            dim0.dim_value = batch
    return proto

def _count_macs_onnx(model_proto):
    inferred = onnx_si.infer_shapes(model_proto)
    shape_map = {}
    for vi in list(inferred.graph.value_info) + list(inferred.graph.input) + list(inferred.graph.output):
        t = vi.type.tensor_type
        if t.HasField("shape"):
            shape_map[vi.name] = [d.dim_value for d in t.shape.dim]
    init_shapes = {init.name: list(init.dims) for init in inferred.graph.initializer}
    def get_shape(name):
        return shape_map.get(name) or init_shapes.get(name, [])

    total_macs = 0
    for node in inferred.graph.node:
        op = node.op_type
        if op == "Conv":
            w, o = get_shape(node.input[1]), get_shape(node.output[0])
            if len(w) >= 4 and len(o) >= 4:
                C_out, C_in_g, Kh, Kw = w[:4]
                _, _, Hout, Wout = o[:4]
                total_macs += Kh * Kw * C_in_g * C_out * Hout * Wout
        elif op == "ConvTranspose":
            w, o = get_shape(node.input[1]), get_shape(node.output[0])
            if len(w) >= 4 and len(o) >= 4:
                C_in, C_out_g, Kh, Kw = w[:4]
                _, C_out, Hout, Wout = o[:4]
                groups = next((int(a.i) for a in node.attribute if a.name == "group"), 1)
                total_macs += Kh * Kw * (C_in // groups) * C_out * Hout * Wout
        elif op == "Gemm":
            a, b = get_shape(node.input[0]), get_shape(node.input[1])
            transA = next((int(x.i) for x in node.attribute if x.name == "transA"), 0)
            transB = next((int(x.i) for x in node.attribute if x.name == "transB"), 0)
            if len(a) >= 2 and len(b) >= 2:
                M = a[-1] if transA else a[-2]
                K = a[-2] if transA else a[-1]
                N = b[-2] if transB else b[-1]
                total_macs += M * N * K
        elif op == "MatMul":
            a, b = get_shape(node.input[0]), get_shape(node.input[1])
            if len(a) >= 2 and len(b) >= 2:
                M, K, N = a[-2], a[-1], b[-1]
                batch = 1
                for d in a[:-2]:
                    batch *= max(int(d), 1)
                total_macs += batch * M * N * K
    return total_macs

def get_gflops_onnx(model_proto, batch=1):
    try:
        return float(_count_macs_onnx(_concretize_batch(model_proto, batch))) / 1e9
    except Exception:
        return float("nan")

PROJECT_ROOT = Path(f"{PROJECT_ROOT}").expanduser().resolve()
EXPORT_DIR = (PROJECT_ROOT / "artifacts" / "onnx_exports").resolve()

ONNX_MODELS = {
    "rtdetr_det"  : EXPORT_DIR / "face_det_rtdetr_train_clean2_op20_b1_640_static_shared_baseline.onnx",
    "mobilenetv3" : PROJECT_ROOT / "artifacts" / "mobilenetv3_best_opset20_dynB.onnx",
    "posterv2"    : PROJECT_ROOT / "artifacts" / "poster_v2_fer2013_dynB.onnx",
    "pcnn"        : PROJECT_ROOT / "artifacts" / "pcnn_best_opset20_dynB.onnx",
    "lanmsff"     : PROJECT_ROOT / "artifacts" / "two_path_massatt_pwfs_opset20.onnx",
}
MODEL_META = {
    "rtdetr_det": {"kind": "detector", "imgsz": 640},
    "mobilenetv3": {"kind": "classifier", "imgsz": 224},
    "posterv2": {"kind": "classifier", "imgsz": 224},
    "pcnn": {"kind": "classifier", "imgsz": 224},
    "lanmsff": {"kind": "classifier", "imgsz": 64},
}

profiles = []
for name, onnx_path in ONNX_MODELS.items():
    onnx_path = Path(onnx_path)
    if not onnx_path.is_file():
        print(f"NOT FOUND: {onnx_path}")
        profiles.append({"Model": name, "Kind": MODEL_META[name]["kind"],
                         "Disk (MB)": None, "Params (M)": None,
                         "VRAM FP32 (MB)": None, "GFLOPs": None})
        continue
    model_proto = onnx.load(str(onnx_path))
    total_params = sum(int(np.prod(init.dims)) for init in model_proto.graph.initializer if len(init.dims) > 0)
    gflops = get_gflops_onnx(model_proto, batch=1)
    profiles.append({
        "Model": name,
        "Kind": MODEL_META[name]["kind"],
        "Disk (MB)": round(onnx_path.stat().st_size / 1e6, 2),
        "Params (M)": round(total_params / 1e6, 3),
        "VRAM FP32 (MB)": round(total_params * 4 / 1e6, 2),
        "GFLOPs": round(gflops, 4) if gflops == gflops else "N/A",
    })

df_onnx = pd.DataFrame(profiles).set_index("Model")
print(df_onnx.to_string())

In [ ]:
import time
import numpy as np, pandas as pd, cv2, torch
from pathlib import Path
import onnxruntime as ort

PROJECT_ROOT = Path(f"{PROJECT_ROOT}").expanduser().resolve()
EXPORT_DIR = (PROJECT_ROOT / "artifacts" / "onnx_exports").resolve()
CLS_TEST_DIR = Path(f"{PROJECT_ROOT}/data/FER2013/test")
DET_TEST_DIR = Path(f"{PROJECT_ROOT}/data/wider_face_w_test/images/test")

WARMUP_PASSES = 10
CLS_CLASSES = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
CLS2IDX = {c: i for i, c in enumerate(CLS_CLASSES)}
IMG_EXT = ('.jpg', '.jpeg', '.png')

ONNX_MODELS = {
    "mobilenetv3": {"path": PROJECT_ROOT / "artifacts" / "mobilenetv3_best_opset20_dynB.onnx",
                    "kind": "classifier", "layout": "nchw", "channels": 3, "imgsz": 224},
    "posterv2": {"path": PROJECT_ROOT / "artifacts" / "poster_v2_fer2013_dynB.onnx",
                 "kind": "classifier", "layout": "nchw", "channels": 3, "imgsz": 224},
    "pcnn": {"path": PROJECT_ROOT / "artifacts" / "pcnn_best_opset20_dynB.onnx",
             "kind": "classifier", "layout": "nchw", "channels": 3, "imgsz": 224},
    "lanmsff": {"path": PROJECT_ROOT / "artifacts" / "two_path_massatt_pwfs_opset20.onnx",
                "kind": "classifier", "layout": "nhwc", "channels": 1, "imgsz": 64},
    "rtdetr_det": {"path": EXPORT_DIR / "face_det_rtdetr_train_clean2_op20_b1_640_static_shared_baseline.onnx",
                   "kind": "detector", "layout": "nchw", "channels": 3, "imgsz": 640},
}

def _make_sess(path):
    avail = ort.get_available_providers()
    providers = (["CUDAExecutionProvider"] if "CUDAExecutionProvider" in avail else []) + ["CPUExecutionProvider"]
    so = ort.SessionOptions()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    so.intra_op_num_threads = 1
    return ort.InferenceSession(path, sess_options=so, providers=providers)

def preprocess_cls(img_path, meta):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    imgsz = meta["imgsz"]
    img = cv2.resize(img, (imgsz, imgsz),
                     interpolation=cv2.INTER_AREA if img.shape[0] > imgsz else cv2.INTER_LINEAR)
    if meta["channels"] == 1:
        img = (cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0)
        img = img[None, None] if meta["layout"] == "nchw" else img[:, :, None][None]
    else:
        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2, 0, 1))[None] if meta["layout"] == "nchw" else img[None]
    return img.astype(np.float32)

def preprocess_det(img_path, imgsz=640):
    img = cv2.imread(str(img_path))
    h0, w0 = img.shape[:2]
    r = min(imgsz / h0, imgsz / w0)
    nh, nw = int(round(h0 * r)), int(round(w0 * r))
    img = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    top = (imgsz - nh) // 2; bot = imgsz - nh - top
    left = (imgsz - nw) // 2; right = imgsz - nw - left
    img = cv2.copyMakeBorder(img, top, bot, left, right, cv2.BORDER_CONSTANT, value=(114, 114, 114))
    img = np.transpose(img.astype(np.float32) / 255.0, (2, 0, 1))[None]
    return img.astype(np.float32)

def postprocess_cls(logits):
    x = logits - logits.max(axis=1, keepdims=True)
    probs = np.exp(x) / np.exp(x).sum(axis=1, keepdims=True)
    return int(np.argmax(probs[0]))

def postprocess_det(raw, conf_thr=0.25):
    return int((raw[0][:, 4] >= conf_thr).sum())

cls_image_list = []
for cls_name in CLS_CLASSES:
    cls_dir = CLS_TEST_DIR / cls_name
    if cls_dir.is_dir():
        for p in sorted(cls_dir.iterdir()):
            if p.suffix.lower() in IMG_EXT:
                cls_image_list.append((p, CLS2IDX[cls_name]))
det_image_list = sorted(p for p in DET_TEST_DIR.rglob("*") if p.suffix.lower() in IMG_EXT)

all_results = {}
for name, meta in ONNX_MODELS.items():
    onnx_path = Path(meta["path"])
    if not onnx_path.is_file():
        print(f"NOT FOUND: {onnx_path}")
        continue
    sess = _make_sess(str(onnx_path))
    inp_name = sess.get_inputs()[0].name
    out_name = sess.get_outputs()[0].name
    is_cls = meta["kind"] == "classifier"
    img_list = cls_image_list if is_cls else [(p, None) for p in det_image_list]

    for img_path, _ in img_list[:WARMUP_PASSES]:
        x = preprocess_cls(img_path, meta) if is_cls else preprocess_det(img_path, meta["imgsz"])
        sess.run([out_name], {inp_name: x})
    torch.cuda.synchronize()

    records = []
    for img_path, true_idx in img_list:
        t0 = time.perf_counter()
        x = preprocess_cls(img_path, meta) if is_cls else preprocess_det(img_path, meta["imgsz"])
        torch.cuda.synchronize(); t_pre = (time.perf_counter() - t0) * 1000
        t0 = time.perf_counter()
        raw = sess.run([out_name], {inp_name: x})[0]
        torch.cuda.synchronize(); t_inf = (time.perf_counter() - t0) * 1000
        t0 = time.perf_counter()
        if is_cls:
            extra = {"correct": int(postprocess_cls(raw) == true_idx)}
        else:
            extra = {"n_dets": postprocess_det(raw)}
        torch.cuda.synchronize(); t_post = (time.perf_counter() - t0) * 1000
        records.append({
            "pre_ms": t_pre, "inf_ms": t_inf, "post_ms": t_post,
            "total_ms": t_pre + t_inf + t_post, **extra,
        })
    all_results[name] = records

summary_rows = []
for name, records in all_results.items():
    n = len(records)
    row = {
        "Model": name, "Kind": ONNX_MODELS[name]["kind"], "Images": n,
        "Avg Pre (ms)": round(sum(r["pre_ms"] for r in records) / n, 3),
        "Avg Inf (ms)": round(sum(r["inf_ms"] for r in records) / n, 3),
        "Avg Post(ms)": round(sum(r["post_ms"] for r in records) / n, 3),
        "Avg Tot (ms)": round(sum(r["total_ms"] for r in records) / n, 3),
    }
    if ONNX_MODELS[name]["kind"] == "classifier":
        row["Top1 Acc (%)"] = round(sum(r["correct"] for r in records) / n * 100, 2)
    else:
        row["Avg dets/img"] = round(sum(r["n_dets"] for r in records) / n, 3)
    summary_rows.append(row)

df_onnx_latency = pd.DataFrame(summary_rows).set_index("Model")
print(df_onnx_latency.to_string())

In [ ]:
import numpy as np, pandas as pd, cv2, torch
from pathlib import Path
import onnxruntime as ort

PROJECT_ROOT = Path(f"{PROJECT_ROOT}").expanduser().resolve()
EXPORT_DIR = (PROJECT_ROOT / "artifacts" / "onnx_exports").resolve()
RTDETR_ONNX = EXPORT_DIR / "face_det_rtdetr_train_clean2_op20_b1_640_static_shared_baseline.onnx"
TEST_IMG_DIR = Path(f"{PROJECT_ROOT}/data/wider_face_w_test/images/test")
TEST_LBL_DIR = Path(f"{PROJECT_ROOT}/data/wider_face_w_test/labels/test")

DET_IMGSZ = 640
MAP_CONF = 0.001
MAP_MAX_DET = 300
IMG_EXT = ('.jpg', '.jpeg', '.png')
MAP_IOUS = [round(x, 2) for x in np.arange(0.50, 0.96, 0.05)]

test_imgs = sorted(p for p in TEST_IMG_DIR.rglob("*") if p.suffix.lower() in IMG_EXT)

def _make_sess(path):
    avail = ort.get_available_providers()
    providers = (["CUDAExecutionProvider"] if "CUDAExecutionProvider" in avail else []) + ["CPUExecutionProvider"]
    so = ort.SessionOptions()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    so.intra_op_num_threads = 1
    return ort.InferenceSession(path, sess_options=so, providers=providers)

sess = _make_sess(str(RTDETR_ONNX))
inp_name = sess.get_inputs()[0].name
out_name = sess.get_outputs()[0].name

def preprocess(img_path, imgsz=640):
    img = cv2.imread(str(img_path))
    h0, w0 = img.shape[:2]
    r = min(imgsz / h0, imgsz / w0)
    nh, nw = int(round(h0 * r)), int(round(w0 * r))
    img = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    top = (imgsz - nh) // 2; bot = imgsz - nh - top
    left = (imgsz - nw) // 2; right = imgsz - nw - left
    img = cv2.copyMakeBorder(img, top, bot, left, right, cv2.BORDER_CONSTANT, value=(114, 114, 114))
    x = np.transpose(img.astype(np.float32) / 255.0, (2, 0, 1))[None]
    return x.astype(np.float32), r, (left, top), (h0, w0)

def postprocess_rtdetr(raw, r, pad, orig_hw, conf_thr, max_det, imgsz=640):
    h0, w0 = orig_hw; dw, dh = pad
    boxes = raw[0]; scores = boxes[:, 4]
    keep = scores >= conf_thr; boxes = boxes[keep]; scores = scores[keep]
    order = np.argsort(-scores)[:max_det]; boxes = boxes[order]; scores = scores[order]
    out = []
    for box, sc in zip(boxes, scores):
        cx, cy, bw, bh = box[0] * imgsz, box[1] * imgsz, box[2] * imgsz, box[3] * imgsz
        x1 = np.clip((cx - bw / 2 - dw) / r, 0, w0); x2 = np.clip((cx + bw / 2 - dw) / r, 0, w0)
        y1 = np.clip((cy - bh / 2 - dh) / r, 0, h0); y2 = np.clip((cy + bh / 2 - dh) / r, 0, h0)
        if x2 > x1 and y2 > y1:
            out.append([float(x1), float(y1), float(x2), float(y2), float(sc)])
    return out

def load_gt_boxes(img_path, img_root, lbl_root):
    txt_path = (lbl_root / img_path.relative_to(img_root)).with_suffix(".txt")
    if not txt_path.exists():
        return []
    img = cv2.imread(str(img_path))
    if img is None:
        return []
    h, w = img.shape[:2]
    boxes = []
    for line in txt_path.read_text(errors="ignore").splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        try:
            xc, yc, bw, bh = map(float, parts[1:5])
        except ValueError:
            continue
        x1 = np.clip((xc - bw / 2) * w, 0, w); x2 = np.clip((xc + bw / 2) * w, 0, w)
        y1 = np.clip((yc - bh / 2) * h, 0, h); y2 = np.clip((yc + bh / 2) * h, 0, h)
        if x2 > x1 and y2 > y1:
            boxes.append([float(x1), float(y1), float(x2), float(y2)])
    return boxes

def box_iou(a, b):
    ix1 = max(a[0], b[0]); iy1 = max(a[1], b[1]); ix2 = min(a[2], b[2]); iy2 = min(a[3], b[3])
    iw = max(0.0, ix2 - ix1); ih = max(0.0, iy2 - iy1); inter = iw * ih
    ua = max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1]); ub = max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])
    union = ua + ub - inter
    return 0.0 if union <= 0 else inter / union

def compute_ap(rec, prec):
    rec = np.concatenate(([0.0], np.asarray(rec), [1.0]))
    prec = np.concatenate(([0.0], np.asarray(prec), [0.0]))
    for i in range(prec.size - 1, 0, -1):
        prec[i - 1] = max(prec[i - 1], prec[i])
    return float(np.mean([np.interp(thr, rec, prec) for thr in np.linspace(0.0, 1.0, 101)]))

def evaluate_map(pred_store, ious):
    total_gt = sum(len(v["gt"]) for v in pred_store.values())
    if total_gt == 0:
        nan = float("nan")
        return {"ap50": nan, "map50_95": nan, "aps": {t: nan for t in ious}, "total_gt": 0, "labeled_imgs": 0}
    labeled_imgs = sum(1 for v in pred_store.values() if v["gt"])
    aps = {}
    for thr in ious:
        all_preds = []
        for img_key, item in pred_store.items():
            for p in item["pred"]:
                all_preds.append((img_key, float(p[4]), p[:4]))
        all_preds.sort(key=lambda z: z[1], reverse=True)
        tp = np.zeros(len(all_preds)); fp = np.zeros(len(all_preds))
        gt_used = {k: [False] * len(v["gt"]) for k, v in pred_store.items()}
        for i, (img_key, score, pbox) in enumerate(all_preds):
            gt_boxes = pred_store[img_key]["gt"]
            if not gt_boxes:
                fp[i] = 1.0; continue
            best_iou = -1.0; best_j = -1
            for j, gbox in enumerate(gt_boxes):
                iou = box_iou(pbox, gbox)
                if iou > best_iou:
                    best_iou = iou; best_j = j
            if best_iou >= thr and not gt_used[img_key][best_j]:
                tp[i] = 1.0; gt_used[img_key][best_j] = True
            else:
                fp[i] = 1.0
        if not len(all_preds):
            aps[thr] = 0.0; continue
        tp_cum = np.cumsum(tp); fp_cum = np.cumsum(fp)
        rec = tp_cum / max(total_gt, 1); prec = tp_cum / np.maximum(tp_cum + fp_cum, 1e-12)
        aps[thr] = compute_ap(rec, prec)
    return {"ap50": aps.get(0.50, float("nan")),
            "map50_95": float(np.mean(list(aps.values()))) if aps else float("nan"),
            "aps": aps, "total_gt": total_gt, "labeled_imgs": labeled_imgs}

for wp in test_imgs[:10]:
    x, _, _, _ = preprocess(wp, DET_IMGSZ)
    sess.run([out_name], {inp_name: x})
torch.cuda.synchronize()

pred_store = {}
total_preds = 0
for img_path in test_imgs:
    img_key = str(img_path.relative_to(TEST_IMG_DIR))
    gt_boxes = load_gt_boxes(img_path, TEST_IMG_DIR, TEST_LBL_DIR)
    x, r, pad, orig_hw = preprocess(img_path, DET_IMGSZ)
    raw = sess.run([out_name], {inp_name: x})[0]
    preds = postprocess_rtdetr(raw, r, pad, orig_hw, conf_thr=MAP_CONF, max_det=MAP_MAX_DET, imgsz=DET_IMGSZ)
    total_preds += len(preds)
    pred_store[img_key] = {"gt": gt_boxes, "pred": preds}

results = evaluate_map(pred_store, MAP_IOUS)

summary_row = pd.DataFrame([{
    "Model": "rtdetr_det", "Dataset": "WiderFace", "Precision": "FP32",
    "AP@50 (%)": round(results["ap50"] * 100, 2),
    "mAP@50-95 (%)": round(results["map50_95"] * 100, 2),
    "GT boxes": results["total_gt"], "GT imgs": results["labeled_imgs"],
    "Total preds": total_preds,
}]).set_index("Model")
print(summary_row.to_string())